In [28]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/malaikatanveer123/iot-network-preprocess/preprocessed_iot_dataset.csv")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 26 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   src_port                    211043 non-null  int64  
 1   dst_port                    211043 non-null  int64  
 2   proto                       211043 non-null  int64  
 3   service                     211043 non-null  int64  
 4   conn_state                  211043 non-null  int64  
 5   dns_qclass                  211043 non-null  int64  
 6   dns_qtype                   211043 non-null  int64  
 7   dns_rcode                   211043 non-null  int64  
 8   dns_AA                      211043 non-null  int64  
 9   ssl_version                 211043 non-null  int64  
 10  ssl_cipher                  211043 non-null  int64  
 11  http_trans_depth            211043 non-null  int64  
 12  http_method                 211043 non-null  int64  
 13  http_version  

Preprocessed dataset from Phase 2 is loaded to perform feature engineering

In [31]:
X = df.drop(['type','label'], axis=1)
y = df['type']   

In [32]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X, y)

rf_importance = pd.Series(rf.feature_importances_, index=X.columns)
rf_importance.sort_values(ascending=False).head(10)

src_ip_bytes_log    0.165220
src_port            0.121101
conn_state          0.117417
dst_port            0.109584
duration_log        0.098314
dst_ip_bytes_log    0.094254
src_pkts_log        0.071023
src_bytes_log       0.066845
dst_bytes_log       0.044145
proto               0.042144
dtype: float64

In [33]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier()
et.fit(X, y)

et_importance = pd.Series(et.feature_importances_, index=X.columns)
et_importance.sort_values(ascending=False).head(10)

conn_state          0.153866
src_ip_bytes_log    0.148498
src_port            0.140475
dst_ip_bytes_log    0.109644
dst_port            0.072232
src_pkts_log        0.069056
src_bytes_log       0.060293
duration_log        0.053984
proto               0.052625
dst_bytes_log       0.043214
dtype: float64

In [34]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

lr_importance = pd.Series(abs(lr.coef_).mean(axis=0), index=X.columns)
lr_importance.sort_values(ascending=False).head(10)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


dns_qtype           0.029557
conn_state          0.016343
dst_port            0.016270
dst_ip_bytes_log    0.012303
dst_bytes_log       0.010291
src_ip_bytes_log    0.009994
src_bytes_log       0.008765
service             0.007468
dns_qclass          0.005921
src_pkts_log        0.004685
dtype: float64

In [35]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier()
lgb.fit(X, y)

lgb_importance = pd.Series(lgb.feature_importances_, index=X.columns)
lgb_importance.sort_values(ascending=False).head(10)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017894 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2374
[LightGBM] [Info] Number of data points in the train set: 211043, number of used features: 23
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -5.309961
[LightGBM] [Info] Start training from score -1.440039
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330


src_port            6842
duration_log        5079
src_ip_bytes_log    3362
dst_port            2809
dst_ip_bytes_log    2736
src_bytes_log       2135
dst_bytes_log       1837
conn_state          1676
src_pkts_log        1253
service              601
dtype: int32

In [38]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X, y)

xgb_importance = pd.Series(xgb.feature_importances_, index=X.columns)
xgb_importance.sort_values(ascending=False).head(10)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:09:11] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


proto               0.313533
conn_state          0.099638
service             0.090630
src_bytes_log       0.070356
dst_port            0.067812
src_pkts_log        0.059165
src_ip_bytes_log    0.054688
ssl_version         0.049594
http_version        0.032801
src_port            0.028404
dtype: float32

Multiple algorithms were used to identify consistently important features across models.

In [39]:
# 1. Traffic Ratios 
df['bytes_ratio'] = df['src_bytes_log'] / (df['dst_bytes_log'] + 1)
df['ip_bytes_ratio'] = df['src_ip_bytes_log'] / (df['dst_ip_bytes_log'] + 1)

# 2. Packet Behavior
df['pkts_ratio'] = df['src_pkts_log'] / (df['src_pkts_log'] + df['dst_bytes_log'] + 1)

# 3. Total Traffic
df['total_bytes'] = df['src_bytes_log'] + df['dst_bytes_log']
df['total_ip_bytes'] = df['src_ip_bytes_log'] + df['dst_ip_bytes_log']

# 4. Traffic Efficiency
df['bytes_per_pkt'] = df['total_bytes'] / (df['src_pkts_log'] + 1)

# 5. Duration-based feature
df['bytes_per_second'] = df['total_bytes'] / (df['duration_log'] + 1)

# 6. Port Interaction
df['port_sum'] = df['src_port'] + df['dst_port']
df['port_diff'] = abs(df['src_port'] - df['dst_port'])

# 7. Suspicious Behavior Indicator
df['high_traffic_flag'] = (df['total_bytes'] > df['total_bytes'].mean()).astype(int)

df.head()

,src_port,dst_port,proto,service,conn_state,dns_qclass,dns_qtype,dns_rcode,dns_AA,ssl_version,...,bytes_ratio,ip_bytes_ratio,pkts_ratio,total_bytes,total_ip_bytes,bytes_per_pkt,bytes_per_second,port_sum,port_diff,high_traffic_flag
0,4444,49178,1,0,0,0,0,0,0,0,...,1.301101,1.252836,0.346176,19.389064,19.841891,3.406761,2.904903,53622,44734,1
1,49180,8080,1,0,1,0,0,0,0,0,...,0.000000,0.842311,0.409384,0.000000,7.683864,0.000000,0.000000,57260,41100,0
2,49180,8080,1,0,1,0,0,0,0,0,...,0.000000,0.842311,0.409384,0.000000,7.683864,0.000000,0.000000,57260,41100,0
3,49180,8080,1,0,1,0,0,0,0,0,...,0.000000,0.825663,0.409384,0.000000,7.605392,0.000000,0.000000,57260,41100,0
4,49180,8080,1,0,1,0,0,0,0,0,...,0.000000,0.842311,0.409384,0.000000,7.683864,0.000000,0.000000,57260,41100,0


New features were created based on network traffic behavior such as byte ratios, packet interactions, and traffic intensity. These features help capture relationships between source and destination communication, which are critical for detecting different types of IoT attacks. 

1. Traffic Ratios

bytes_ratio

This feature represents the ratio between source bytes and destination bytes. It helps identify asymmetric communication patterns, which are common in attacks such as DoS where one side sends significantly more data.

ip_bytes_ratio

This captures the ratio of IP-level bytes between source and destination. It provides deeper insight into packet-level traffic imbalance and abnormal communication behavior.

2. Packet Behavior

pkts_ratio

This feature represents the relationship between packet count and overall traffic. It helps detect irregular packet transmission patterns that may indicate scanning or flooding attacks.

3. Total Traffic

total_bytes

This feature represents the total volume of data exchanged between source and destination. High values may indicate heavy traffic attacks such as Distributed Denial of Service (DDoS).

total_ip_bytes

This represents the total IP-level traffic. It provides a more complete view of network load and helps in identifying abnormal traffic spikes.

4. Traffic Efficiency

bytes_per_pkt

This feature measures the average size of data per packet. Abnormal values may indicate inefficient or malicious packet usage, which is often seen in certain types of attacks.

5. Duration-Based Feature

bytes_per_second

This feature captures the rate of data transfer over time. High transmission rates in short durations may indicate flooding attacks or abnormal bursts in traffic.

6. Port Interaction

port_sum

This feature combines source and destination ports to capture overall port activity, which may help identify unusual communication patterns.

port_diff

This feature represents the difference between source and destination ports. Large differences or unusual combinations can indicate suspicious behavior or non-standard communication.

7. Suspicious Behavior Indicator

high_traffic_flag

This binary feature identifies whether a connection has higher-than-average traffic. It helps in quickly distinguishing potential abnormal or attack-related traffic from normal behavior.

In [40]:
X_new = df.drop(['type','label'], axis=1)

from lightgbm import LGBMClassifier
lgb = LGBMClassifier()

lgb.fit(X_new, df['type'])

import pandas as pd
pd.Series(lgb.feature_importances_, index=X_new.columns).sort_values(ascending=False).head(15)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024871 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4671
[LightGBM] [Info] Number of data points in the train set: 211043, number of used features: 33
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -5.309961
[LightGBM] [Info] Start training from score -1.440039
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330


duration_log        3303
src_port            3090
bytes_per_second    2987
dst_port            1990
src_ip_bytes_log    1860
port_sum            1773
port_diff           1619
total_ip_bytes      1527
ip_bytes_ratio      1341
bytes_per_pkt       1297
src_bytes_log       1292
bytes_ratio         1159
dst_ip_bytes_log    1103
conn_state          1077
pkts_ratio           858
dtype: int32

In [41]:
importance = pd.Series(lgb.feature_importances_, index=X_new.columns)

# Low importance features
low_features = importance[importance < 800].index

print(low_features)

# Drop them
df = df.drop(low_features, axis=1)

df.shape

Index(['proto', 'service', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA',
       'ssl_version', 'ssl_cipher', 'http_trans_depth', 'http_method',
       'http_version', 'weird_name', 'http_response_body_len_log',
       'missed_bytes_log', 'http_request_body_len_log', 'src_pkts_log',
       'total_bytes', 'high_traffic_flag'],
      dtype='object')


(211043, 18)

In [ ]:
Features with very low importance were removed to reduce noise and improve model performance. Only meaningful and impactful features were retained.

In [42]:
X_final = df.drop(['type','label'], axis=1)

lgb.fit(X_final, df['type'])

pd.Series(lgb.feature_importances_, index=X_final.columns).sort_values(ascending=False).head(10)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012379 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3838
[LightGBM] [Info] Number of data points in the train set: 211043, number of used features: 16
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -5.309961
[LightGBM] [Info] Start training from score -1.440039
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330
[LightGBM] [Info] Start training from score -2.356330


duration_log        3536
bytes_per_second    3140
src_port            2978
dst_port            2279
src_ip_bytes_log    2145
ip_bytes_ratio      1633
total_ip_bytes      1612
bytes_per_pkt       1591
port_sum            1551
src_bytes_log       1532
dtype: int32

In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = df.drop(['type','label'], axis=1).columns

df[features] = scaler.fit_transform(df[features])

In [44]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5)

df['cluster'] = kmeans.fit_predict(df[features])

K-Means clustering was applied to discover hidden patterns in network traffic and create an additional feature representing cluster membership.

In [46]:
df.to_csv("feature_engineered_iot_dataset.csv", index=False)

In [47]:
from IPython.display import FileLink
FileLink('feature_engineered_iot_dataset.csv')

/kaggle/working/feature_engineered_iot_dataset.csv